# Use Case 9 — RL Training Layer: Rewards, Eval Metrics, Reward Model

GPU-free reward shaping + trajectory eval metrics, then a real GPU run: TRL `RewardTrainer` (LoRA) trains a preference reward model.

## 1. Reward shaping + trajectory lifecycle (GPU-free)

In [ ]:
"""
Lifecycle + rewards — `collect`, `evaluate_agentic`, and reward shaping.
========================================================================

The other case studies use `infer` / `collect_rollout` / `evaluate`. This one closes the loop on
the remaining lifecycle surface and the reward pillar that GRPO training consumes:

  - `Project.collect(tasks)`       -> run a batch of tasks, keep every EventLog
  - `Project.evaluate_agentic()`   -> the agentic metrics report (tac/ter/arr/scsr/rad/lcf)
  - `answer_match(log, expected)`  -> a spine-native reward straight off an EventLog
  - `REWARD_REGISTRY` + `combine_rewards(...)` -> a weighted TRL-style reward_func the GRPO
    trainer would call on candidate completions

GPU-free; the reward functions are the exact ones a real GRPO run would score with.
"""
from agenttune.agentic import (
    Project, DictToolHarness, ReActStrategy,
    REWARD_REGISTRY, combine_rewards, answer_match,
)

In [2]:
def calc(x=0, y=0):
    return {"result": x + y}

In [3]:
def policy(state):
    if state.step == 0:
        return {"name": "calc", "arguments": {"x": 19, "y": 23}, "thought": "add the operands"}
    return {"name": "finish", "arguments": {"answer": "42"}, "thought": "report the sum"}

In [4]:
def main():
    proj = Project(strategy=ReActStrategy(policy, max_steps=4),
                   harness=DictToolHarness({"calc": calc}, max_steps=4))

    # 1) collect() — run a batch of tasks and keep every EventLog.
    logs = proj.collect(["what is 19 + 23?", "what is 19 + 23?"])
    print(f"[collect]  {len(logs)} episodes collected, tiers={[l.tier for l in logs]}")

    # 2) evaluate_agentic() — the agentic metrics report over a task set.
    report = proj.evaluate_agentic(["what is 19 + 23?"])
    print(f"[evaluate] n={report['n']}  metrics -> {report['metrics']}")

    # 3) answer_match — a reward computed directly off a spine EventLog.
    print(f"[reward]   answer_match(log, '42') -> {answer_match(logs[0], '42')}")

    # 4) Reward shaping for GRPO: pick funcs from the registry, weight them, and score
    #    candidate completions exactly as the trainer's rollout would.
    print(f"[registry] {len(REWARD_REGISTRY)} built-in reward funcs, e.g. "
          f"{sorted(REWARD_REGISTRY)[:3]}")
    reward_fn = combine_rewards(
        [REWARD_REGISTRY["reward_correct_answer"], REWARD_REGISTRY["reward_concise_answer"]],
        weights=[0.7, 0.3],
    )
    candidates = [
        [{"role": "assistant", "content": "42"}],                       # correct + concise
        [{"role": "assistant", "content": "the answer is 99 " + "and " * 90 + "done"}],  # wrong + rambling
    ]
    scores = reward_fn(candidates, answer=["42", "42"])
    print(f"[shape]    weighted reward over 2 candidates -> {[round(s, 2) for s in scores]} "
          f"(correct+concise beats wrong+long)")

In [5]:
if __name__ == "__main__":
    main()

[collect]  2 episodes collected, tiers=['light', 'light']


[evaluate] n=1  metrics -> {'tac': 0.0, 'ter': 1.0, 'arr': 0.0, 'scsr': 1.0, 'rad': 0.0, 'lcf': 0.0}
[reward]   answer_match(log, '42') -> 1.0
[registry] 18 built-in reward funcs, e.g. ['answer_correctness_reward', 'answer_format_reward', 'computation_reward']
[shape]    weighted reward over 2 candidates -> [1.14, 0.0] (correct+concise beats wrong+long)


## 2. Real reward-model training (GPU)

In [ ]:
"""
REAL reward-model training — trains the scorer that RL optimises against, on the GPU.
=====================================================================================

GRPO/PPO need a reward. Reward *functions* are programmatic; the other real path is a learned
*reward model* — a scalar scorer trained from preference data to rank a good answer above a bad
one. This trains one for real with TRL's `RewardTrainer` (LoRA) on `{prompt, chosen, rejected}`
preference data, the same shape the self-heal loop emits.

A reward model is `AutoModelForSequenceClassification` with one output. After training it should
score `chosen` above `rejected` on held-out pairs — that margin is the whole point.

Requires a GPU and SmolLM2-360M in the local Hugging Face cache.
"""
import os
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

In [7]:
import sys
sys.modules["vllm"] = None  # env vllm is ABI-broken vs torch; TRL imports it eagerly

In [8]:
import torch
from datasets import Dataset
from peft import LoraConfig
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from trl import RewardTrainer, RewardConfig

In [9]:
from agenttune.agentic import build_dataset
from agenttune.decide.closed_loop.contracts import TrainingExample

In [10]:
MODEL = "HuggingFaceTB/SmolLM2-360M-Instruct"
OUT_DIR = "/tmp/agenttune-reward-model"

In [11]:
def preference(text, label):
    """Corrective preference: the contract answer (chosen) ranked over a prose answer (rejected)."""
    return TrainingExample(
        trajectory_id=f"rm-{text[:6]}", original_failure_type="format_violation", root_cause="prose",
        prompt=[{"role": "user", "content": f"Classify the sentiment: '{text}'"}],
        chosen=[{"role": "assistant", "content": f"SENTIMENT={label}"}],
        rejected=[{"role": "assistant", "content": f"The sentiment of this review is {label}."}],
    )

In [12]:
def score(model, tok, prompt, answer):
    text = tok.apply_chat_template(
        [{"role": "user", "content": prompt}, {"role": "assistant", "content": answer}],
        tokenize=False)
    enc = tok(text, return_tensors="pt", truncation=True, max_length=128).to(model.device)
    with torch.no_grad():
        return model(**enc).logits[0, 0].item()

In [13]:
def main():
    print(f"[gpu]     {torch.cuda.get_device_name(0)}  (cuda: {torch.cuda.is_available()})")

    reviews = [
        ("I love this product, best purchase ever", "positive"),
        ("absolutely fantastic experience", "positive"),
        ("exceeded my expectations", "positive"),
        ("delighted with the quality", "positive"),
        ("this is the worst thing I've bought", "negative"),
        ("terrible, broke on day one", "negative"),
        ("support was rude and unhelpful", "negative"),
        ("waste of money, deeply disappointed", "negative"),
        ("it arrived on time, works as described", "neutral"),
        ("standard packaging, nothing special", "neutral"),
        ("does the job, no strong feelings", "neutral"),
        ("average product, meets the basics", "neutral"),
    ]
    rows = build_dataset([preference(t, lbl) for t, lbl in reviews])
    print(f"[data]    {len(rows)} preference rows (schema {list(rows[0].keys())}) — chosen=contract, rejected=prose")

    tok = AutoTokenizer.from_pretrained(MODEL)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    rm = AutoModelForSequenceClassification.from_pretrained(
        MODEL, num_labels=1, torch_dtype=torch.bfloat16).to("cuda")
    rm.config.pad_token_id = tok.pad_token_id

    probe = ("Classify the sentiment: 'terrible, broke on day one'", "SENTIMENT=negative",
             "The sentiment of this review is negative.")

    # Mean pairwise margin: how far the RM scores chosen above rejected, averaged over the set.
    # (Accuracy saturates — a random head ranks these easy pairs right by chance; margin is the
    # signal that actually moves as the model learns to score the correction confidently.)
    def mean_margin(model):
        return sum(score(model, tok, r["prompt"][0]["content"], r["chosen"][0]["content"])
                   - score(model, tok, r["prompt"][0]["content"], r["rejected"][0]["content"])
                   for r in rows) / len(rows)

    before = (score(rm, tok, probe[0], probe[1]), score(rm, tok, probe[0], probe[2]))
    margin_before = mean_margin(rm)  # measured before the trainer wraps rm in-place

    cfg = RewardConfig(
        output_dir=OUT_DIR, num_train_epochs=10, per_device_train_batch_size=4,
        learning_rate=1e-4, max_length=128, logging_steps=2, save_strategy="no",
        report_to=[], bf16=True,
    )
    trainer = RewardTrainer(
        model=rm, args=cfg, train_dataset=Dataset.from_list(rows), processing_class=tok,
        peft_config=LoraConfig(r=16, lora_alpha=32, task_type="SEQ_CLS",
                               target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]),
    )
    trainer.train()

    margin_after = mean_margin(trainer.model)
    after = (score(trainer.model, tok, probe[0], probe[1]), score(trainer.model, tok, probe[0], probe[2]))

    print(f"[rm]      mean chosen-minus-rejected margin over {len(rows)} pairs "
          f"{margin_before:+.2f} -> {margin_after:+.2f}")
    print(f"[probe]   {probe[0]}")
    print(f"[before]  reward(chosen)={before[0]:+.2f}  reward(rejected)={before[1]:+.2f}  "
          f"chosen_wins={before[0] > before[1]}")
    print(f"[after]   reward(chosen)={after[0]:+.2f}  reward(rejected)={after[1]:+.2f}  "
          f"chosen_wins={after[0] > after[1]}")
    print(f"[verdict] reward model ranks the contract answer above prose, margin "
          f"{after[0] - after[1]:+.2f} (was {before[0] - before[1]:+.2f})")

In [14]:
if __name__ == "__main__":
    main()

[gpu]     NVIDIA L4  (cuda: True)
[data]    12 preference rows (schema ['prompt', 'chosen', 'rejected']) — chosen=contract, rejected=prose


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[transformers] LlamaForSequenceClassification LOAD REPORT from: HuggingFaceTB/SmolLM2-360M-Instruct
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Tokenizing train dataset:   0%|          | 0/12 [00:00<?, ? examples/s]

Filtering train >128 tokens:   0%|          | 0/12 [00:00<?, ? examples/s]

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
2,0.782959
4,0.722715
6,0.647722
8,0.568169
10,0.495240
12,0.423903
14,0.359092
16,0.296326
18,0.247796
20,0.196205


[rm]      mean chosen-minus-rejected margin over 12 pairs -0.19 -> +2.56
[probe]   Classify the sentiment: 'terrible, broke on day one'
[before]  reward(chosen)=-0.82  reward(rejected)=-0.64  chosen_wins=False
[after]   reward(chosen)=+0.09  reward(rejected)=-2.53  chosen_wins=True
[verdict] reward model ranks the contract answer above prose, margin +2.62 (was -0.18)
